In [1]:
import re
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec, KeyedVectors
from gensim.models.phrases import Phrases, Phraser
import gensim.downloader as api
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [2]:
## load original imdb data (https://ai.stanford.edu/%7Eamaas/data/sentiment/)

import os

def read_imdb_data(data_dir):
    texts = []
    labels = []

    for split in ["train", "test"]:
        for label_name, label in [("pos", 1), ("neg", 0)]:
            folder = os.path.join(data_dir, split, label_name)
            for filename in os.listdir(folder):
                if filename.endswith(".txt"):
                    with open(os.path.join(folder, filename), "r", encoding="utf-8") as f:
                        texts.append(f.read())
                        labels.append(label)
    return texts, np.array(labels)

data_dir = "../datasets/aclImdb"
texts, labels = read_imdb_data(data_dir)

print(f"Loaded {len(texts)} reviews, {sum(labels)} positive, {len(labels)-sum(labels)} negative")

Loaded 50000 reviews, 25000 positive, 25000 negative


In [3]:
texts[:5]

['For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagine a movie where Joe Piscopo is actually funny! Maureen Stapleton is a scene stealer. The Moroni character is an absolute scream. Watch for Alan "The Skipper" Hale jr. as a police Sgt.',
 'Bizarre horror movie filled with famous faces but stolen by Cristina Raines (later of TV\'s "Flamingo Road") as a pretty but somewhat unstable model with a gummy smile who is slated to pay for her attempted suicides by guarding the Gateway to Hell! The scenes with Raines modeling are very well captured, the mood music is perfect, Deborah Raffin is charming as Cristina\'s pal, but when Raines moves into a creepy Brooklyn Heights brownstone (inhabited by a blind priest on the top floor), things really start cooking. The neighbors, including a fantastically wicked Burgess Meredith and kinky couple Sylvia Miles & Beverly D\'Angelo, are a diabolical lot, and Eli Wallach is great fun as a wily police detec

In [4]:
# Key Mappings for lables
# 1 -> positive
# 0 -> negative

In [5]:
labels[:10]

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [6]:
# basic preprocessor


def preprocess_google_news(text):
    r"""
    It removes characters that are not alphanumeric or underscores from the input text.
    This preprocessing step is useful for preparing text data for word embedding models like Google News Word2Vec.
    `\w`: letters, digits and underscores
    `\s`: whitespace characters
    `_"`: underscore

    Example: "Great movie!!! 10/10." becomes "Great movie 1010".

    text.split() splits the cleaned text at whitespace.

    Result: ["Great", "movie", "1010"].:

    Args:
        text (str): The input text to preprocess.

    Returns:
        list: A list of preprocessed words from the input text.
    """
    text = re.sub(r"[^\w\s_]", "", text)
    return text.split()

In [7]:
## tokenize and phrase detection

tokenized_texts = [preprocess_google_news(t) for t in texts]
tokenized_texts[0][:10]

['For', 'a', 'movie', 'that', 'gets', 'no', 'respect', 'there', 'sure', 'are']

In [8]:
phrases = Phrases(tokenized_texts, min_count=5, threshold=10)
bigram = Phraser(phrases)

tokenized_texts = [bigram[sent] for sent in tokenized_texts]

In [16]:
print(tokenized_texts[:10])

[['For', 'a', 'movie', 'that', 'gets', 'no', 'respect', 'there', 'sure', 'are', 'a', 'lot', 'of', 'memorable_quotes', 'listed', 'for', 'this', 'gem', 'Imagine', 'a', 'movie', 'where', 'Joe_Piscopo', 'is', 'actually', 'funny', 'Maureen_Stapleton', 'is', 'a', 'scene_stealer', 'The', 'Moroni', 'character', 'is', 'an_absolute', 'scream', 'Watch', 'for', 'Alan', 'The', 'Skipper', 'Hale', 'jr', 'as', 'a', 'police', 'Sgt'], ['Bizarre', 'horror', 'movie', 'filled_with', 'famous', 'faces', 'but', 'stolen', 'by', 'Cristina_Raines', 'later', 'of', 'TVs', 'Flamingo', 'Road', 'as', 'a', 'pretty', 'but', 'somewhat', 'unstable', 'model', 'with', 'a', 'gummy', 'smile', 'who', 'is', 'slated', 'to', 'pay', 'for', 'her', 'attempted', 'suicides', 'by', 'guarding', 'the', 'Gateway', 'to', 'Hell', 'The', 'scenes', 'with', 'Raines', 'modeling', 'are', 'very_well', 'captured', 'the', 'mood', 'music', 'is', 'perfect', 'Deborah_Raffin', 'is', 'charming', 'as', 'Cristinas', 'pal', 'but', 'when', 'Raines', 'moves

In [9]:
!pip3 install gensim

In [13]:
## download embeddings
w2v_pretrained = api.load('word2vec-google-news-300')

In [14]:
## MODEL OFF-THE-SHELF WORD2VEC

In [19]:
# Compute OOV + Coverage

total_tokens = 0
oov_tokens = 0

for sentence in tokenized_texts:
    for word in sentence:
        total_tokens += 1
        if word not in w2v_pretrained:
            oov_tokens += 1

oov_rate_pre = oov_tokens / total_tokens
coverage_pre = 1 - oov_rate_pre

print("\nFROZEN EMBEDDINGS")
print("Total tokens:", total_tokens)
print("OOV tokens:", oov_tokens)
print(f"OOV rate: {oov_rate_pre:.2%}")
print(f"Coverage: {coverage_pre:.2%}")


FROZEN EMBEDDINGS
Total tokens: 10697674
OOV tokens: 2097699
OOV rate: 19.61%
Coverage: 80.39%


In [20]:
# build vocab

vocab = {}
for sentence in tokenized_texts:
    for word in sentence:
        if word in w2v_pretrained and word not in vocab:
            vocab[word] = len(vocab) + 1

In [25]:
(vocab)

{'For': 1,
 'movie': 2,
 'that': 3,
 'gets': 4,
 'no': 5,
 'respect': 6,
 'there': 7,
 'sure': 8,
 'are': 9,
 'lot': 10,
 'listed': 11,
 'for': 12,
 'this': 13,
 'gem': 14,
 'Imagine': 15,
 'where': 16,
 'Joe_Piscopo': 17,
 'is': 18,
 'actually': 19,
 'funny': 20,
 'Maureen_Stapleton': 21,
 'scene_stealer': 22,
 'The': 23,
 'Moroni': 24,
 'character': 25,
 'scream': 26,
 'Watch': 27,
 'Alan': 28,
 'Skipper': 29,
 'Hale': 30,
 'jr': 31,
 'as': 32,
 'police': 33,
 'Sgt': 34,
 'Bizarre': 35,
 'horror': 36,
 'famous': 37,
 'faces': 38,
 'but': 39,
 'stolen': 40,
 'by': 41,
 'later': 42,
 'TVs': 43,
 'Flamingo': 44,
 'Road': 45,
 'pretty': 46,
 'somewhat': 47,
 'unstable': 48,
 'model': 49,
 'with': 50,
 'gummy': 51,
 'smile': 52,
 'who': 53,
 'slated': 54,
 'pay': 55,
 'her': 56,
 'attempted': 57,
 'suicides': 58,
 'guarding': 59,
 'the': 60,
 'Gateway': 61,
 'Hell': 62,
 'scenes': 63,
 'Raines': 64,
 'modeling': 65,
 'captured': 66,
 'mood': 67,
 'music': 68,
 'perfect': 69,
 'charming': 

In [26]:
# convert to sequence

MAX_LEN = 200

X = [
    [vocab[w] for w in sentence if w in vocab]
    for sentence in tokenized_texts
]

X = pad_sequences(X, maxlen=MAX_LEN, padding="post")

In [27]:
# train and test split

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

In [28]:
# build embedding matrix (from off the shelf word2vec)

embedding_dim = 300
embedding_matrix = np.zeros((len(vocab) + 1, embedding_dim))

for word, i in vocab.items():
    embedding_matrix[i] = w2v_pretrained[word]

In [29]:
model = Sequential([
    Embedding(
        input_dim=len(vocab) + 1,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=False
    ),
    LSTM(128),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

/Users/shubham.kumar.3/Desktop/Deep-Learning/.venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │    33,784,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,784,500 (128.88 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 33,784,500 (128.88 MB)

In [30]:
# train model
history = model.fit(
    X_train,
    y_train,
    batch_size=64,
    epochs=5,
    validation_data=(X_test, y_test)
)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 72s 114ms/step - accuracy: 0.5381 - loss: 0.6902 - val_accuracy: 0.5290 - val_loss: 0.6888
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 71s 114ms/step - accuracy: 0.5284 - loss: 0.6874 - val_accuracy: 0.5401 - val_loss: 0.6803
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 71s 113ms/step - accuracy: 0.5439 - loss: 0.6836 - val_accuracy: 0.5566 - val_loss: 0.6733
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 71s 114ms/step - accuracy: 0.5615 - loss: 0.6710 - val_accuracy: 0.5581 - val_loss: 0.6689
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 71s 113ms/step - accuracy: 0.5876 - loss: 0.6598 - val_accuracy: 0.5579 - val_loss: 0.6667


In [31]:
# model 1 accuracy

loss, acc = model.evaluate(X_test, y_test)

print("\nRESULTS")
print("Frozen Accuracy:", acc)

print("\nCoverage")
print(f"Frozen Coverage: {coverage_pre:.2%}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.5579 - loss: 0.6667

RESULTS
Frozen Accuracy: 0.5579000115394592

Coverage
Frozen Coverage: 80.39%
